In [76]:
import os
import random
import shutil
from pathlib import Path
from collections import Counter
import torch

from PIL import Image, ImageEnhance, ImageFilter
from tqdm import tqdm

In [77]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "train"
BALANCED_DIR = PROJECT_ROOT / "data" / "balanced" / "train"

CLASSES = ["real", "deepfake", "ai_generated", "edited"]
TARGET_COUNT = 500
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("BALANCED_DIR:", BALANCED_DIR)

PROJECT_ROOT: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject
TRAIN_DIR: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\processed\train
BALANCED_DIR: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\balanced\train


In [78]:
def count_images_by_class(base_dir, classes):
    counts = {}
    for cls in classes:
        class_dir = base_dir / cls
        if class_dir.exists():
            image_files = [f for f in class_dir.iterdir() if f.is_file()]
            counts[cls] = len(image_files)
        else:
            counts[cls] = 0
    return counts

original_counts = count_images_by_class(TRAIN_DIR, CLASSES)

print("Original class distribution:")
for cls, count in original_counts.items():
    print(f"{cls}: {count}")

Original class distribution:
real: 7099
deepfake: 7378
ai_generated: 111
edited: 176


In [79]:
max_count = max(original_counts.values())
print("Target count per class:", max_count)

Target count per class: 7378


In [80]:
BALANCED_DIR.mkdir(parents=True, exist_ok=True)

for cls in CLASSES:
    (BALANCED_DIR / cls).mkdir(parents=True, exist_ok=True)

print("Balanced output folders created successfully.")

Balanced output folders created successfully.


In [81]:
def augment_image(img):
    aug_type = random.choice([
        "flip",
        "rotate",
        "brightness",
        "contrast",
        "blur"
    ])

    if aug_type == "flip":
        return img.transpose(Image.FLIP_LEFT_RIGHT)

    elif aug_type == "rotate":
        angle = random.choice([-10, -7, -5, 5, 7, 10])
        return img.rotate(angle)

    elif aug_type == "brightness":
        enhancer = ImageEnhance.Brightness(img)
        return enhancer.enhance(random.uniform(0.8, 1.2))

    elif aug_type == "contrast":
        enhancer = ImageEnhance.Contrast(img)
        return enhancer.enhance(random.uniform(0.8, 1.2))

    elif aug_type == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))

    return img

In [82]:
def copy_original_images(src_dir, dst_dir):
    image_files = [f for f in src_dir.iterdir() if f.is_file()]
    
    for img_path in tqdm(image_files, desc=f"Copying {src_dir.name}"):
        shutil.copy2(img_path, dst_dir / img_path.name)

    return image_files

In [83]:
def generate_augmented_images(image_files, dst_dir, target_count):
    current_count = len([f for f in dst_dir.iterdir() if f.is_file()])
    needed = target_count - current_count

    if needed <= 0:
        return

    for i in tqdm(range(needed), desc=f"Augmenting {dst_dir.name}"):
        src_path = random.choice(image_files)

        try:
            with Image.open(src_path) as img:
                img = img.convert("RGB")
                aug_img = augment_image(img)
        except Exception as e:
            print(f"Error processing {src_path.name}: {e}")

In [84]:
def generate_augmented_images(image_files, dst_dir, target_count):
    current_count = len([f for f in dst_dir.iterdir() if f.is_file()])
    needed = target_count - current_count

    if needed <= 0:
        return

    for i in tqdm(range(needed), desc=f"Augmenting {dst_dir.name}"):
        src_path = random.choice(image_files)

        try:
            with Image.open(src_path) as img:
                img = img.convert("RGB")
                aug_img = augment_image(img)
                new_name = f"{src_path.stem}_aug_{i}{src_path.suffix}"
                aug_img.save(dst_dir / new_name)
        except Exception as e:
            print(f"Error processing {src_path.name}: {e}")

In [85]:
from collections import Counter

class_to_idx = {
    "real": 0,
    "deepfake": 1,
    "edited": 2,
    "ai_generated": 3
}

idx_to_class = {v: k for k, v in class_to_idx.items()}

train_counts = {
    "real": 500,
    "deepfake": 500,
    "edited": 176,
    "ai_generated": 111
}

print("Training counts:")
for cls, cnt in train_counts.items():
    print(f"{cls}: {cnt}")

Training counts:
real: 500
deepfake: 500
edited: 176
ai_generated: 111


In [86]:
import torch 
class_weights = torch.tensor([
    0.64,  # real
    0.64,  # deepfake
    1.83,  # edited
    2.90   # ai_generated
], dtype=torch.float32)

In [87]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

print("Device:", device)
print("Weighted CrossEntropyLoss ready")
print("Weights on device:", class_weights.to(device))

Device: cuda
Weighted CrossEntropyLoss ready
Weights on device: tensor([0.6400, 0.6400, 1.8300, 2.9000], device='cuda:0')


In [88]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 1
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [89]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

torch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
CUDA device count: 1
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [90]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

print("Device:", device)
print("Weighted CrossEntropyLoss ready")
print("Weights on device:", class_weights.to(device))

Device: cuda
Weighted CrossEntropyLoss ready
Weights on device: tensor([0.6400, 0.6400, 1.8300, 2.9000], device='cuda:0')


In [91]:
import torch.backends.cudnn as cudnn

cudnn.benchmark = True

print("cuDNN benchmark enabled:", cudnn.benchmark)

cuDNN benchmark enabled: True


In [92]:
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train"

print("Using dataset from:", TRAIN_PATH)
print("Exists:", TRAIN_PATH.exists())

Using dataset from: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\processed\train
Exists: True


In [93]:
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)

print("Train dataset size:", len(train_dataset))
print("Class to index mapping:", train_dataset.class_to_idx)
print("Classes:", train_dataset.classes)

Train dataset size: 14764
Class to index mapping: {'ai_generated': 0, 'deepfake': 1, 'edited': 2, 'real': 3}
Classes: ['ai_generated', 'deepfake', 'edited', 'real']


In [94]:
import torch
import torch.nn as nn

base_weight_map = {
    "real": 0.64,
    "deepfake": 0.64,
    "edited": 1.83,
    "ai_generated": 2.90
}

ordered_weights = torch.tensor(
    [base_weight_map[c] for c in train_dataset.classes],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(weight=ordered_weights.to(device))

print("Dataset class order:", train_dataset.classes)
print("Ordered weights:", ordered_weights)

Dataset class order: ['ai_generated', 'deepfake', 'edited', 'real']
Ordered weights: tensor([2.9000, 0.6400, 1.8300, 0.6400])


In [95]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("First 10 labels:", labels[:10].tolist())

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
First 10 labels: [3, 3, 3, 3, 1, 1, 1, 3, 3, 3]


In [96]:
VAL_PATH = PROJECT_ROOT / "data" / "processed" / "val"

print("Using validation data from:", VAL_PATH)
print("Exists:", VAL_PATH.exists())

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_dataset = datasets.ImageFolder(VAL_PATH, transform=val_transform)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Validation dataset size:", len(val_dataset))
print("Validation class mapping:", val_dataset.class_to_idx)

Using validation data from: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\processed\val
Exists: True
Validation dataset size: 3168
Validation class mapping: {'ai_generated': 0, 'deepfake': 1, 'edited': 2, 'real': 3}


In [97]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [98]:
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 4)

model = model.to(device)

print("Model moved to:", device)
print("Final classifier:", model.classifier)

Model moved to: cuda
Final classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


In [40]:
import torch.optim as optim

LEARNING_RATE = 1e-3
EPOCHS = 10

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Optimizer:", optimizer)
print("Scheduler: ReduceLROnPlateau")
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Scheduler: ReduceLROnPlateau
Learning rate: 0.001
Epochs: 10


In [99]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [100]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_labels, all_preds

In [101]:
# ============================================================
# CELL: Run training
# ============================================================

import copy

num_epochs = 10

best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_preds, val_labels = validate_one_epoch(
        model, val_loader, criterion, device
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_efficientnet_b0.pth")
        print("✅ Best model saved.")

# Load best weights after training
model.load_state_dict(best_model_wts)
print("\nTraining complete. Best validation loss:", best_val_loss)


Epoch [1/10]
--------------------------------------------------


KeyboardInterrupt: 

In [102]:
# ============================================================
# CELL: Check class balance
# ============================================================

from collections import Counter

train_counts = Counter(train_dataset.targets)
val_counts = Counter(val_dataset.targets)

print("Train class counts:")
for k, v in sorted(train_counts.items()):
    print(f"{k} ({idx_to_class[k]}): {v}")

print("\nValidation class counts:")
for k, v in sorted(val_counts.items()):
    print(f"{k} ({idx_to_class[k]}): {v}")

Train class counts:
0 (real): 111
1 (deepfake): 7378
2 (edited): 176
3 (ai_generated): 7099

Validation class counts:
0 (real): 25
1 (deepfake): 1582
2 (edited): 39
3 (ai_generated): 1522


In [104]:
print("Train dir:", TRAIN_DIR)
print("Val dir:", VAL_PATH)
print("Train classes:", train_dataset.classes)
print("Class to idx:", train_dataset.class_to_idx)

Train dir: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\processed\train
Val dir: c:\Extra stuff\capstone\DeepfakeProject\DeepfakeProject\data\processed\val
Train classes: ['ai_generated', 'deepfake', 'edited', 'real']
Class to idx: {'ai_generated': 0, 'deepfake': 1, 'edited': 2, 'real': 3}
